# Stage 4.5 — TrackNetV2 Fine-Tune

Fine-tunes Andrew Dettor's pickleball TrackNetV2 weights on user-labeled
frames from four match videos. Designed for Google Colab with T4 GPU (free tier).

**Prerequisites in `MyDrive/pickleball/`:**
- `tracknet_v2_dettor.pt` — Dettor's converted weights
- `_tracknet_model.py` — vendored from `stages/track_ball/`
- `training_data.zip` — output of `prepare_training_data.py`, zipped

**Runtime:** Runtime > Change runtime type > T4 GPU.

**Outputs (back to Drive):**
- `tracknet_v2_finetuned_v<N>.pt` — best-val-loss weights
- `finetune_log_v<N>.json` — per-epoch loss + hyperparameters

Bump `VERSION` in Cell 2 for each new training run.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime > Change runtime type > T4 GPU.')
print(f'Device: {torch.cuda.get_device_name(0)}')


In [ ]:
import hashlib
from pathlib import Path

VERSION = 1  # bump for each new training run

DRIVE_ROOT = Path('/content/drive/MyDrive/pickleball')
WEIGHTS_IN = DRIVE_ROOT / 'tracknet_v2_dettor.pt'
MODEL_PY   = DRIVE_ROOT / '_tracknet_model.py'
DATA_ZIP   = DRIVE_ROOT / 'training_data.zip'

WEIGHTS_OUT = DRIVE_ROOT / f'tracknet_v2_finetuned_v{VERSION}.pt'
LOG_OUT     = DRIVE_ROOT / f'finetune_log_v{VERSION}.json'

LOCAL_DATA = Path('/content/training')

for p in (WEIGHTS_IN, MODEL_PY, DATA_ZIP):
    if not p.exists():
        raise FileNotFoundError(f'Missing input: {p}')

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

print(f'_tracknet_model.py SHA-256: {sha256(MODEL_PY)}')
print(f'tracknet_v2_dettor.pt SHA-256: {sha256(WEIGHTS_IN)}')
print(f'training_data.zip size: {DATA_ZIP.stat().st_size / 1e9:.2f} GB')
print(f'Output weights -> {WEIGHTS_OUT}')
print(f'Output log     -> {LOG_OUT}')
if WEIGHTS_OUT.exists():
    print(f'WARNING: {WEIGHTS_OUT} already exists; will be overwritten on save.')


In [ ]:
import json
import time
import zipfile

if LOCAL_DATA.exists() and any(LOCAL_DATA.iterdir()):
    print(f'{LOCAL_DATA} already populated; skipping unzip')
else:
    t0 = time.time()
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(LOCAL_DATA)
    print(f'unzipped in {time.time() - t0:.1f}s')

# Locate metadata.json (zip layout may vary by how user packaged it)
metadata_path = None
for candidate in [
    LOCAL_DATA / 'metadata.json',
    LOCAL_DATA / 'training' / 'metadata.json',
    LOCAL_DATA / 'data' / 'training' / 'metadata.json',
]:
    if candidate.exists():
        LOCAL_DATA = candidate.parent
        metadata_path = candidate
        break
if metadata_path is None:
    for depth in ('*/metadata.json', '*/*/metadata.json'):
        hits = list(LOCAL_DATA.glob(depth))
        if len(hits) == 1:
            metadata_path = hits[0]
            LOCAL_DATA = metadata_path.parent
            break

if metadata_path is None:
    raise FileNotFoundError(f'metadata.json not found under {LOCAL_DATA}')

with open(metadata_path) as f:
    metadata = json.load(f)

n_train_files = len(list((LOCAL_DATA / 'train').glob('*.npz')))
n_val_files   = len(list((LOCAL_DATA / 'val').glob('*.npz')))

print(f'data dir:      {LOCAL_DATA}')
print(f'metadata says: train={metadata["n_train"]}, val={metadata["n_val"]}')
print(f'on disk:       train={n_train_files}, val={n_val_files}')
print(f'val source:    {metadata["val_source_video"]}')
print(f'heatmap sigma: {metadata["heatmap_sigma"]}')
assert n_train_files == metadata['n_train'], 'train file count mismatch'
assert n_val_files == metadata['n_val'], 'val file count mismatch'


## Load TrackNet and Dettor's weights

Imports `_tracknet_model.py` from Drive. Loads Dettor's weights with
`strict=True` — same as Stage 4's loader — then runs a forward pass on a
zero tensor to verify shape and output range.


In [ ]:
import sys

if str(DRIVE_ROOT) not in sys.path:
    sys.path.insert(0, str(DRIVE_ROOT))
if '_tracknet_model' in sys.modules:
    del sys.modules['_tracknet_model']
from _tracknet_model import TrackNet  # noqa: E402

DEVICE = torch.device('cuda')

model = TrackNet(in_channels=9, out_channels=3).to(DEVICE)
state = torch.load(WEIGHTS_IN, map_location=DEVICE, weights_only=True)
model.load_state_dict(state, strict=True)
print(f'loaded Dettor weights from {WEIGHTS_IN}')

n_params = sum(p.numel() for p in model.parameters())
print(f'parameters: {n_params:,}')

model.eval()
with torch.no_grad():
    dummy = torch.zeros(1, 9, 288, 512, device=DEVICE)
    y = model(dummy)
print(f'forward pass: input {tuple(dummy.shape)} -> output {tuple(y.shape)}')
print(f'output range on zeros: [{y.min().item():.4f}, {y.max().item():.4f}]')
assert y.shape == (1, 3, 288, 512), f'unexpected output shape {y.shape}'
assert 0.0 <= y.min().item() and y.max().item() <= 1.0, \
    f'output not in [0,1] (model not post-sigmoid?): min={y.min().item()} max={y.max().item()}'
print('sanity checks passed')


## Dataset and augmentation

Loads `.npz` samples from `prepare_training_data.py`. Casts uint8 input to
float32 and divides by 255; casts float16 target to float32. Training set
gets random horizontal flip (50%), brightness +-15%, contrast +-15%, and
per-color RGB shift +-5%. Augmentations are consistent across the 3 frames
so temporal coherence is preserved. No spatial cropping (would break the
camera-position assumption per the contract).


In [ ]:
import random
import numpy as np
from torch.utils.data import Dataset, DataLoader

class BallDataset(Dataset):
    def __init__(self, root, split, augment):
        self.root = Path(root) / split
        self.files = sorted(self.root.glob('*.npz'))
        if not self.files:
            raise FileNotFoundError(f'no .npz files in {self.root}')
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        d = np.load(self.files[idx])
        x = d['input'].astype(np.float32) / 255.0
        y = d['target'].astype(np.float32)

        if self.augment:
            if random.random() < 0.5:
                x = x[:, :, ::-1].copy()
                y = y[:, :, ::-1].copy()

            b = 1.0 + (random.random() - 0.5) * 0.3
            x = np.clip(x * b, 0, 1)

            c = 1.0 + (random.random() - 0.5) * 0.3
            mean = x.mean()
            x = np.clip((x - mean) * c + mean, 0, 1)

            # Channel layout: [R0,G0,B0,R1,G1,B1,R2,G2,B2]
            for color in range(3):
                shift = (random.random() - 0.5) * 0.1
                x[color::3] = np.clip(x[color::3] + shift, 0, 1)

        return torch.from_numpy(x), torch.from_numpy(y)

train_ds = BallDataset(LOCAL_DATA, 'train', augment=True)
val_ds   = BallDataset(LOCAL_DATA, 'val',   augment=False)
print(f'train: {len(train_ds)} samples')
print(f'val:   {len(val_ds)} samples')


## Hyperparameters, loss, dataloaders

Adam at lr=1e-4 (lower than from-scratch — we are fine-tuning).
Weighted BCE with `pos_weight=100`: the Gaussian peak covers ~50 pixels out
of 288*512=147,456, so positives are ~3000x rarer. A literal inverse-frequency
weight would blow up; 100 is a TrackNet-community-typical compromise.


In [ ]:
BATCH_SIZE = 4
LR = 1e-4
POS_WEIGHT = 100.0
EPOCHS = 30
PATIENCE = 5
WALL_BUDGET_SECONDS = 2.5 * 3600

def weighted_bce(pred, target, pos_weight):
    eps = 1e-7
    pred = pred.clamp(eps, 1 - eps)
    loss_pos = -target * torch.log(pred) * pos_weight
    loss_neg = -(1 - target) * torch.log(1 - pred)
    return (loss_pos + loss_neg).mean()

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True, persistent_workers=True,
)

steps_per_train_epoch = len(train_loader)
steps_per_val_epoch   = len(val_loader)
print(f'train steps/epoch: {steps_per_train_epoch}')
print(f'val   steps/epoch: {steps_per_val_epoch}')
print(f'epochs: up to {EPOCHS}, patience: {PATIENCE}, wall budget: {WALL_BUDGET_SECONDS/3600:.2f}h')


## Training loop

Per epoch: train through all samples, then validate. Track best val loss;
stop early if val loss does not improve for `PATIENCE` consecutive epochs.
Hard wall-clock guard prevents training from being killed mid-epoch by Colab.


In [ ]:
import copy

best_val_loss = float('inf')
best_state = None
best_epoch = None
patience_counter = 0
log_entries = []
stopped_reason = None
training_start = time.time()

for epoch in range(1, EPOCHS + 1):
    elapsed_before = time.time() - training_start
    if elapsed_before > WALL_BUDGET_SECONDS:
        stopped_reason = f'wall_budget exceeded before epoch {epoch} ({elapsed_before:.0f}s)'
        print(stopped_reason)
        break

    epoch_start = time.time()

    model.train()
    tloss_sum = 0.0
    n = 0
    for step, (x, y) in enumerate(train_loader):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        pred = model(x)
        loss = weighted_bce(pred, y, POS_WEIGHT)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        tloss_sum += loss.item() * x.size(0)
        n += x.size(0)
        if step % 200 == 0:
            print(f'  epoch {epoch} step {step}/{steps_per_train_epoch} loss={loss.item():.4f}')
    train_loss = tloss_sum / n

    model.eval()
    vloss_sum = 0.0
    n = 0
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            pred = model(x)
            loss = weighted_bce(pred, y, POS_WEIGHT)
            vloss_sum += loss.item() * x.size(0)
            n += x.size(0)
    val_loss = vloss_sum / n

    epoch_seconds = time.time() - epoch_start
    cumulative = time.time() - training_start

    entry = {
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'epoch_seconds': epoch_seconds,
        'cumulative_seconds': cumulative,
    }
    log_entries.append(entry)

    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        patience_counter = 0
        marker = ' (best)'
    else:
        patience_counter += 1
        marker = f' (no improvement, patience {patience_counter}/{PATIENCE})'

    print(f'epoch {epoch}: train={train_loss:.4f} val={val_loss:.4f} ({epoch_seconds:.0f}s){marker}')

    if patience_counter >= PATIENCE:
        stopped_reason = f'early stopping at epoch {epoch}'
        print(stopped_reason)
        break

if stopped_reason is None:
    stopped_reason = f'completed all {EPOCHS} epochs'
    print(stopped_reason)

if best_state is None:
    raise RuntimeError('no best state captured; training produced no usable checkpoint')

print(f'\nbest val loss: {best_val_loss:.4f} at epoch {best_epoch}')


## Save outputs to Drive


In [ ]:
import datetime as dt

torch.save(best_state, WEIGHTS_OUT)
print(f'wrote {WEIGHTS_OUT} ({WEIGHTS_OUT.stat().st_size / 1e6:.1f} MB)')

log_payload = {
    'schema_version': 1,
    'version': VERSION,
    'weights_path': str(WEIGHTS_OUT),
    'best_val_loss': best_val_loss,
    'best_epoch': best_epoch,
    'epochs_trained': len(log_entries),
    'stopped_reason': stopped_reason,
    'hyperparameters': {
        'batch_size': BATCH_SIZE,
        'learning_rate': LR,
        'pos_weight': POS_WEIGHT,
        'patience': PATIENCE,
        'max_epochs': EPOCHS,
        'wall_budget_seconds': WALL_BUDGET_SECONDS,
        'augmentation': 'hflip 50%, brightness +-15%, contrast +-15%, RGB shift +-5%',
        'optimizer': 'Adam',
    },
    'dataset': {
        'n_train': metadata['n_train'],
        'n_val': metadata['n_val'],
        'heatmap_sigma': metadata['heatmap_sigma'],
        'val_source_video': metadata['val_source_video'],
        'train_source_videos': metadata['train_source_videos'],
    },
    'per_epoch': log_entries,
    'completed_at_utc': dt.datetime.now(dt.UTC).isoformat().replace('+00:00', 'Z'),
}

with open(LOG_OUT, 'w') as f:
    json.dump(log_payload, f, indent=2)
print(f'wrote {LOG_OUT}')


## Next steps

1. Download `tracknet_v2_finetuned_v<N>.pt` from Drive to local `data/models/`.
2. Run Stage 4.5 validation locally (next file to be delivered: `validate.py`).
3. Re-run Stage 4 smoke test on `test_clip` with the new `--weights` path.
4. If both pass, Stage 4.5 is done.
